# Train a PixelCNN on Fashion-MNIST

Move from the four-pixel blackboard example to a real-image autoregressive model with exact likelihood, masked convolutions, GPU or Apple-Metal acceleration, saved weights, unconditional sampling, and conditional completion.

**Classroom pattern:** predict first, run one cell, inspect the intermediate state, then change one parameter. All random seeds are fixed for the first run.

## Before you run

- Use the environment from `README.md`; no lecturer-specific Conda environment is required.
- Set `LECTURE2_QUICK=1` before starting Jupyter for the CPU-safe verification path.
- CUDA is selected first, then Apple MPS, then CPU. A separate MLX installation is not required.
- Read the prediction prompt before each experiment. The fixed-seed result is a mechanism check, not a quality benchmark.
- Every generated file stays inside `fallbacks/` or `checkpoints/`. No cell uploads or publishes anything.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "demo_models.py").exists():
    candidate = ROOT / "lecture 2" / "demos"
    if candidate.exists():
        ROOT = candidate
sys.path.insert(0, str(ROOT))
(ROOT / "fallbacks").mkdir(exist_ok=True)
print(f"Demo root: {ROOT}")

from dataclasses import asdict
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from pixelcnn_training import load_pixelcnn_for_demo, resolve_profile, sample_pixelcnn, train_pixelcnn
torch.manual_seed(29)

## 1. Choose an explicit training profile

The profile is part of the experiment, not a hidden machine-specific detail. `smoke` verifies the pipeline, `local` is a laptop/lab baseline, and `h100` is the prepared full-quality run for one 80 GB NVIDIA H100. The H100 profile uses the complete 28×28 Fashion-MNIST training split, a 128-channel/15-layer model, batch size 1024, 50 epochs, BF16, channels-last tensors, `torch.compile`, validation, resumable latest checkpoints, and a separate best checkpoint.

For a long H100 job, the terminal command is safer than leaving a browser kernel open:

```bash
python tools/train_pixelcnn.py --profile h100 --resume auto
```

Afterwards, set `LECTURE2_PIXELCNN_MODE=load` to use the H100 checkpoint on the presentation Mac without retraining. Optional `LECTURE2_PIXELCNN_SAMPLE_COUNT` and `LECTURE2_PIXELCNN_COMPLETION_COUNT` variables keep live sampling short. Training never publishes anything. Inspect the saved checkpoint and samples first.

In [ ]:
QUICK_MODE = os.getenv("LECTURE2_QUICK", "0") == "1"
PROFILE_NAME = os.getenv("LECTURE2_PIXELCNN_PROFILE", "smoke" if QUICK_MODE else "local")
profile = resolve_profile(PROFILE_NAME)
resume = os.getenv("LECTURE2_PIXELCNN_RESUME", "auto" if PROFILE_NAME == "h100" else "none")
MODE = os.getenv("LECTURE2_PIXELCNN_MODE", "train")
checkpoint_override = os.getenv("LECTURE2_PIXELCNN_CHECKPOINT")
artifact_suffix = "_smoke" if PROFILE_NAME == "smoke" else "_local" if PROFILE_NAME == "local" else ""
print(json.dumps(asdict(profile), indent=2))
print("mode:", MODE, "resume policy:", resume)

## 2. Train, validate, and checkpoint

Fashion-MNIST contains 28×28 grayscale images of clothing. The shared trainer performs a seeded train/validation split, evaluates bits per dimension after every epoch, writes an atomic `latest` checkpoint for resuming, and updates the canonical checkpoint only when validation likelihood improves. After the final epoch it evaluates the untouched 10,000-image test split. The first completed epoch gives a machine-specific runtime estimate; this is more reliable than assuming that every H100 software stack has identical throughput.

In [ ]:
if MODE == "load":
    checkpoint_path = Path(checkpoint_override) if checkpoint_override else ROOT / "checkpoints" / "pixelcnn_fashionmnist.pt"
    result = load_pixelcnn_for_demo(ROOT, checkpoint_path)
elif MODE == "train":
    result = train_pixelcnn(ROOT, profile, resume=resume, progress=True)
else:
    raise ValueError("LECTURE2_PIXELCNN_MODE must be 'train' or 'load'")
model = result.model
device = result.device
data = result.data
profile = result.profile
LEVELS = profile.levels
inputs = data.test_inputs
targets = data.test_targets
labels_np = data.test_labels
CHECKPOINT_PATH = result.best_checkpoint
print({
    "best_checkpoint": str(CHECKPOINT_PATH.relative_to(ROOT)),
    "best_validation_bpd": round(result.best_validation_bpd, 4),
    "test_bpd": round(result.test_bpd, 4),
    "elapsed_minutes": round(result.elapsed_seconds / 60, 2),
})

fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, image, label in zip(axes.flat, inputs[:16], labels_np[:16]):
    ax.imshow(image[0], cmap="gray", vmin=-1, vmax=1)
    ax.set_title(str(label)); ax.axis("off")
fig.suptitle("Fashion-MNIST examples; labels are shown only for reference, not used by PixelCNN")
fig.tight_layout(); plt.show()

## 3. What the masked-convolution objective computed

At every spatial position, the model predicts one of 16 intensity values. Mask A blocks the current pixel and all future pixels. Later Mask B layers may see the current hidden state but still cannot see future pixels. Training evaluates all observed targets in parallel. The validation and test numbers above are normalised negative log-likelihoods in bits per dimension; lower is better.

## 4. Exact likelihood on a complete image

Exact means the normalised categorical probability for every observed pixel can be evaluated and summed. It does not mean that generation is deterministic or that the most likely image will look most plausible.

In [ ]:
model.eval()
with torch.no_grad():
    example_x = inputs[:1].to(device)
    example_target = targets[:1].to(device)
    logits = model(example_x)
    nll_nats = nn.functional.cross_entropy(logits, example_target, reduction="sum")
    bits_per_dimension = nll_nats / example_target.numel() / np.log(2)
print(f"image NLL: {float(nll_nats.cpu()):.2f} nats")
print(f"normalised: {float(bits_per_dimension.cpu()):.3f} bits/dimension")

## 5. Sequential sampling

Generation is intentionally slow: each sampled pixel becomes context for the next model call. Training used one forward pass per batch; a 28×28 sample requires 784 sequential model evaluations. The number of parallel sample chains changes throughput, not the dependency depth.

In [ ]:
sample_count = int(os.getenv("LECTURE2_PIXELCNN_SAMPLE_COUNT", profile.sample_count))
samples = sample_pixelcnn(
    model,
    device,
    LEVELS,
    tuple(targets.shape[1:]),
    count=sample_count,
    temperature=0.9,
    seed=30,
)
fig, axes = plt.subplots(1, sample_count, figsize=(2.2 * sample_count, 2.3), squeeze=False)
for ax, sample in zip(axes.flat, samples):
    ax.imshow(sample[0], cmap="gray", vmin=-1, vmax=1); ax.axis("off")
fig.suptitle("SMOKE PROFILE: pipeline test, not model-quality evidence" if PROFILE_NAME == "smoke" else "Unconditional sequential samples")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / f"06_pixelcnn_samples{artifact_suffix}.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Conditional completion

The top half is observed. PixelCNN samples the lower half from p(x_missing | x_observed). Different seeds can yield different valid completions for the same prefix.

In [ ]:
completion_count = int(os.getenv("LECTURE2_PIXELCNN_COMPLETION_COUNT", profile.completion_count))
prefix = inputs[:1].repeat(completion_count, 1, 1, 1)
known_rows = prefix.shape[-2] // 2
completions = sample_pixelcnn(
    model,
    device,
    LEVELS,
    tuple(targets.shape[1:]),
    count=completion_count,
    temperature=0.85,
    seed=45,
    prefix=prefix,
    known_rows=known_rows,
)
fig, axes = plt.subplots(1, completion_count + 1, figsize=(2.2 * (completion_count + 1), 2.4))
axes[0].imshow(prefix[0, 0], cmap="gray", vmin=-1, vmax=1); axes[0].set_title("original")
for index, image in enumerate(completions, start=1):
    axes[index].imshow(image[0], cmap="gray", vmin=-1, vmax=1); axes[index].set_title(f"completion {index}")
for ax in axes: ax.axis("off")
fig.suptitle("SMOKE PROFILE: completion mechanics only" if PROFILE_NAME == "smoke" else "One observed prefix, multiple sampled futures")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / f"06_pixelcnn_completions{artifact_suffix}.png", dpi=180, bbox_inches="tight")
plt.show()

## 7. Student experiments

1. Compare exact image likelihood with visual plausibility.
2. Change sampling temperature while keeping the seed fixed.
3. Change the known prefix height.
4. Measure sampling time as image resolution grows.
5. Explain why training is parallel but sampling is sequential.

## 8. Inspect first; publish separately

Only the H100 profile writes the canonical `checkpoints/pixelcnn_fashionmnist.pt`. The `local` and `smoke` profiles use different filenames and cannot overwrite it. Inspect validation/test likelihood, samples, completions, and the recorded profile before publishing. Publication is a separate explicit command and is never triggered by this notebook:

```bash
python tools/publish_huggingface.py joergsimon/fh-joanneum-pixelcnn-fashionmnist
```

## Interpretation check and optional extension

Compare bits per dimension with visual plausibility and sampling time. Explain why a model can assign exact likelihoods while still producing weak samples.

**Optional extension:** compare the `local` and supplied full checkpoint on held-out likelihood, completion diversity, and wall-clock sampling cost.

Record the changed parameter, the fixed controls, the observed evidence, and one limitation.